# 09 — Risk-Managed Backtester (Kelly + Vol-Targeting + DD Breaker)

**Goal:** Run the risk-managed backtest using the best Optuna model from Notebook 08. Replaces the 100% all-in/all-out backtester with a production-grade risk engine.

**Three risk layers:**
1. **Kelly Fraction Sizing**: `position = (prob - threshold) / (1 - threshold)`, capped at 1.0
2. **Volatility Targeting**: scale inversely to 20-day realized vol → target 20% annualized
3. **Drawdown Circuit Breaker**: flatten all positions and halt trading if DD ≤ -15%

In [ ]:
import sys, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'Data' / 'cryptonews.csv').exists() or (ROOT / 'notebooks').exists():
        break
    if ROOT == ROOT.parent:
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

INTERIM = ROOT / 'notebooks' / 'interim'
OUTPUTS = ROOT / 'outputs'

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

with (INTERIM / 'features_for_lstm.pkl').open('rb') as f:
    bundle = pickle.load(f)

with (OUTPUTS / 'best_optuna_params.json').open() as f:
    best = json.load(f)

hp = best['best_params']
print(f'Best Optuna params: {hp}')
print(f'Best OOF Sharpe from search: {best["best_oof_sharpe"]:+.4f}')

## 9.1 Train the final model with best Optuna params
Train on train+val combined, evaluate on test.

In [ ]:
import tensorflow as tf
from tensorflow.keras import callbacks
from src.models.lstm import build_lstm_with_params
from sklearn.utils.class_weight import compute_class_weight

tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)
np.random.seed(42)

train_x, train_y = bundle['train_x'], bundle['train_y']
val_x, val_y = bundle['val_x'], bundle['val_y']
test_x, test_y = bundle['test_x'], bundle['test_y']
test_close = bundle['test_close']
test_dates = pd.to_datetime(bundle['test_dates'])
n_features = train_x.shape[-1]

X_full = np.concatenate([train_x, val_x], axis=0)
y_full = np.concatenate([train_y, val_y], axis=0)

classes = np.unique(y_full)
weights = compute_class_weight('balanced', classes=classes, y=y_full)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}

model = build_lstm_with_params(
    lr=hp['lr'], units=hp['units'], dropout=hp['dropout'],
    num_layers=hp['num_layers'], n_features=n_features,
)
print(f'Training on {len(X_full)} samples (train+val) ...')
history = model.fit(
    X_full, y_full, validation_split=0.15,
    epochs=30, batch_size=32, verbose=0,
    class_weight=class_weight,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=7, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5),
    ],
)
print(f'Trained {len(history.history["loss"])} epochs')

test_prob = model.predict(test_x, verbose=0).ravel()
print(f'Test predictions: mean={test_prob.mean():.3f} std={test_prob.std():.3f}')
model.save(INTERIM / 'best_optuna_model.keras')

## 9.2 Run the risk-managed backtest

In [ ]:
from src.backtest.risk_managed import risk_managed_backtest, compare_strategies

rm = risk_managed_backtest(
    prob=test_prob, close=test_close,
    threshold=0.5, fee=0.001,
    target_annual_vol=0.20, vol_lookback=20,
    max_drawdown_pct=0.15,
)

summary = rm.to_summary_dict()
for k, v in summary.items():
    print(f'  {k:30s} {v}')

## 9.3 Strategy comparison: Simple vs Risk-Managed vs Buy & Hold

In [ ]:
comparison = compare_strategies(test_prob, test_close, threshold=0.5, fee=0.001)
comparison
comparison.to_csv(OUTPUTS / 'strategy_comparison.csv', index=False)

## 9.4 Visualize equity curve and position sizing

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
try:
    fm.fontManager.addfont('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
except Exception:
    pass
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(3, 1, figsize=(14, 10), constrained_layout=True, sharex=True)

# Equity curve
axes[0].plot(test_dates, rm.equity, label='Risk-Managed', color='#0f766e', lw=2)
bh_equity = np.cumprod(1 + np.diff(test_close, prepend=test_close[0]) / np.concatenate([[test_close[0]], test_close[:-1]]))
axes[0].plot(test_dates, bh_equity / bh_equity[0], label='Buy & Hold', color='#f59e0b', lw=1.5, alpha=0.7)
axes[0].set_title('Equity Curves', fontweight='bold')
axes[0].legend()
axes[0].set_ylabel('Equity')

# Position size
axes[1].fill_between(test_dates, rm.positions, 0, color='#3b82f6', alpha=0.5)
axes[1].set_title('Position Size (Kelly × Vol-Target)', fontweight='bold')
axes[1].set_ylabel('Position')
axes[1].set_ylim(0, 1)

# Drawdown
running_max = np.maximum.accumulate(rm.equity)
dd = (rm.equity - running_max) / running_max
axes[2].fill_between(test_dates, dd * 100, 0, color='#dc2626', alpha=0.4)
axes[2].axhline(-15, ls='--', c='red', lw=1, label='Circuit breaker (-15%)')
axes[2].set_title('Drawdown', fontweight='bold')
axes[2].set_ylabel('Drawdown %')
axes[2].legend()
axes[2].tick_params(axis='x', rotation=20)

plt.savefig(OUTPUTS / 'risk_managed_backtest.png', dpi=140, bbox_inches='tight')
plt.show()

## 9.5 Save results

In [ ]:
pd.DataFrame([summary]).to_csv(OUTPUTS / 'risk_managed_backtest_results.csv', index=False)

equity_df = pd.DataFrame({
    'date': test_dates,
    'equity': rm.equity,
    'position': rm.positions,
    'kelly_fraction': rm.kelly_fraction,
    'vol_target_factor': rm.vol_target_factor,
    'raw_prob': rm.raw_signal,
})
equity_df.to_csv(OUTPUTS / 'risk_managed_equity_curve.csv', index=False)

print(f'Saved:')
print(f'  outputs/risk_managed_backtest_results.csv')
print(f'  outputs/risk_managed_equity_curve.csv')
print(f'  outputs/strategy_comparison.csv')

## 9.6 Summary
- Trained final LSTM with best Optuna params (lr=5.6e-4, units=32, dropout=0, layers=1).
- Risk-managed backtest applies Kelly sizing + vol-targeting + DD circuit breaker.
- The model's low confidence (probabilities near 0.5) produces small Kelly positions — the risk engine correctly takes minimal risk.
- Circuit breaker was not triggered (max DD stayed above -15%).
- All three strategies compared in `outputs/strategy_comparison.csv`.